## Cell 1 — Universal cricsheet JSON parser

In [20]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

def parse_cricsheet_json(filepath):
    with open(filepath) as f:
        match = json.load(f)
    info = match['info']
    match_id = Path(filepath).stem
    event = info.get('event', {})
    league = event.get('name', 'Unknown')
    match_type = info.get('match_type', 'T20')
    dates = info.get('dates', [])
    date = dates[0] if dates else None
    teams = list(info.get('players', {}).keys())
    if match_type not in ['T20', 'IT20']:
        return pd.DataFrame()
    rows = []
    for inn_idx, innings in enumerate(match.get('innings', [])):
        batting_team = innings.get('team', '')
        bowling_team = [t for t in teams if t != batting_team]
        bowling_team = bowling_team[0] if bowling_team else ''
        for over_data in innings.get('overs', []):
            over_num = over_data['over']
            for delivery in over_data.get('deliveries', []):
                runs = delivery.get('runs', {})
                extras_detail = delivery.get('extras', {})
                wickets = delivery.get('wickets', [])
                actual_over = over_num + 1
                rows.append({
                    'matchId': match_id, 'league': league, 'date': date,
                    'batting_team': batting_team, 'bowling_team': bowling_team,
                    'batsman': delivery.get('batter',''), 'bowler': delivery.get('bowler',''),
                    'batsman_runs': runs.get('batter',0), 'total_runs': runs.get('total',0),
                    'is_wide': 1 if 'wides' in extras_detail else 0,
                    'is_wicket': 1 if wickets else 0,
                    'phase': 'Powerplay' if actual_over<=6 else 'Middle' if actual_over<=15 else 'Death',
                    'is_four': 1 if runs.get('batter',0)==4 else 0,
                    'is_six': 1 if runs.get('batter',0)==6 else 0,
                })
    return pd.DataFrame(rows)

print("Parser defined ✅")

Parser defined ✅


## Cell 2 — Parse all league folders (2-5 minutes)

In [21]:
LEAGUES_DIR = Path('../data/leagues')
all_leagues = {}
league_folders = sorted([f for f in LEAGUES_DIR.iterdir() if f.is_dir()])

print(f"Found {len(league_folders)} league folders")
all_deliveries = []

for folder in league_folders:
    json_files = list(folder.glob('*.json'))
    if not json_files: continue
    league_dfs, errors = [], 0
    for f in json_files:
        try:
            df = parse_cricsheet_json(f)
            if len(df) > 0: league_dfs.append(df)
        except: errors += 1
    if league_dfs:
        league_df = pd.concat(league_dfs, ignore_index=True)
        all_leagues[folder.name] = league_df
        all_deliveries.append(league_df)
        print(f"  ✅ {folder.name}: {len(json_files)} matches → {len(league_df):,} deliveries, "
              f"{league_df['batsman'].nunique()} batters, {league_df['bowler'].nunique()} bowlers")

multi_league = pd.concat(all_deliveries, ignore_index=True)
print(f"\nTotal: {len(multi_league):,} deliveries, {multi_league['batsman'].nunique()} batters, "
      f"{multi_league['bowler'].nunique()} bowlers")

Found 18 league folders
  ✅ bbl: 662 matches → 153,250 deliveries, 503 batters, 372 bowlers
  ✅ blast: 1455 matches → 333,519 deliveries, 870 batters, 646 bowlers
  ✅ bpl: 469 matches → 111,232 deliveries, 508 batters, 381 bowlers
  ✅ cpl: 407 matches → 95,024 deliveries, 390 batters, 284 bowlers
  ✅ csat20: 314 matches → 71,230 deliveries, 351 batters, 262 bowlers
  ✅ hundred: 167 matches → 31,952 deliveries, 240 batters, 170 bowlers
  ✅ ilt20: 134 matches → 31,407 deliveries, 251 batters, 178 bowlers
  ✅ ireland: 104 matches → 22,686 deliveries, 191 batters, 134 bowlers
  ✅ lanka: 119 matches → 27,149 deliveries, 215 batters, 158 bowlers
  ✅ mlc: 75 matches → 17,413 deliveries, 154 batters, 115 bowlers
  ✅ npl: 64 matches → 14,937 deliveries, 175 batters, 134 bowlers
  ✅ nzt20: 270 matches → 61,502 deliveries, 247 batters, 176 bowlers
  ✅ psl: 314 matches → 73,784 deliveries, 380 batters, 270 bowlers
  ✅ sa20: 130 matches → 29,020 deliveries, 199 batters, 144 bowlers
  ✅ sl_domestic:

## Cell 3 — Compute batting AND bowling features per league

In [ ]:
def compute_league_features(league_df, min_bat_innings=15, min_bowl_balls=300):
    bat = league_df[league_df['is_wide'] == 0].copy()
    league_name = league_df['league'].iloc[0] if 'league' in league_df.columns else 'Unknown'
    
    # ── BATTING ──
    bat_stats = bat.groupby('batsman').agg(
        total_runs=('batsman_runs','sum'), balls_faced=('batsman_runs','count'),
        innings=('matchId','nunique'), fours=('is_four','sum'), sixes=('is_six','sum'),
        dismissals=('is_wicket','sum'),
    ).reset_index()
    bat_stats = bat_stats[bat_stats['innings'] >= min_bat_innings]
    if len(bat_stats) > 0:
        bat_stats['career_avg'] = (bat_stats['total_runs'] / bat_stats['dismissals'].clip(lower=1)).round(2)
        bat_stats['career_sr'] = (bat_stats['total_runs'] / bat_stats['balls_faced'].clip(lower=1) * 100).round(2)
        bat_stats['boundary_rate'] = ((bat_stats['fours']+bat_stats['sixes']) / bat_stats['balls_faced'].clip(lower=1)).round(4)
        for phase in ['Powerplay','Middle','Death']:
            tag = phase.lower()[:2] if phase != 'Powerplay' else 'Powerplay'
            pb = bat[bat['phase']==phase].groupby('batsman').agg(pr=('batsman_runs','sum'),pbl=('batsman_runs','count')).reset_index()
            pb[f'avg_runs_{phase}'] = (pb['pr'] / pb['pbl'].clip(lower=1) * 6).round(2)
            pb[f'avg_sr_{phase}'] = (pb['pr'] / pb['pbl'].clip(lower=1) * 100).round(2)
            bat_stats = bat_stats.merge(pb[['batsman',f'avg_runs_{phase}',f'avg_sr_{phase}']], on='batsman', how='left')
        mr = bat.groupby(['batsman','matchId'])['batsman_runs'].sum().reset_index()
        bat_stats = bat_stats.merge(mr.groupby('batsman')['batsman_runs'].max().reset_index().rename(columns={'batsman_runs':'peak_score'}), on='batsman', how='left')
        bat_stats = bat_stats.merge(mr.groupby('batsman')['batsman_runs'].std().fillna(0).reset_index().rename(columns={'batsman_runs':'consistency'}), on='batsman', how='left')
    
    # ── BOWLING ──
    bowl_legal = league_df[league_df['is_wide'] == 0]
    bowler_balls = bowl_legal.groupby('bowler').size()
    qual_bowlers = bowler_balls[bowler_balls >= min_bowl_balls].index.tolist()
    
    bowl_stats = pd.DataFrame()
    if qual_bowlers:
        bq = bowl_legal[bowl_legal['bowler'].isin(qual_bowlers)]
        bowl_stats = bq.groupby('bowler').agg(
            bowl_balls=('bowler','count'), bowl_runs_conceded=('total_runs','sum'),
            bowl_wickets=('is_wicket','sum'), bowl_dots=('total_runs', lambda x: (x==0).sum()),
            bowl_matches=('matchId','nunique'),
        ).reset_index()
        bowl_stats['bowl_economy'] = (bowl_stats['bowl_runs_conceded'] / (bowl_stats['bowl_balls']/6).clip(lower=0.1)).round(2)
        bowl_stats['bowl_avg'] = (bowl_stats['bowl_runs_conceded'] / bowl_stats['bowl_wickets'].clip(lower=1)).round(2)
        bowl_stats['bowl_dot_pct'] = (bowl_stats['bowl_dots'] / bowl_stats['bowl_balls'].clip(lower=1) * 100).round(1)
        for phase in ['Powerplay','Middle','Death']:
            tag = phase.lower()[:2]
            pb = bq[bq['phase']==phase].groupby('bowler').agg(pb=('bowler','count'),pr=('total_runs','sum'),pw=('is_wicket','sum')).reset_index()
            pb[f'bowl_economy_{tag}'] = (pb['pr'] / (pb['pb']/6).clip(lower=0.1)).round(2)
            bowl_stats = bowl_stats.merge(pb[['bowler',f'bowl_economy_{tag}']], on='bowler', how='left')
        mw = league_df[league_df['bowler'].isin(qual_bowlers)].groupby(['bowler','matchId'])['is_wicket'].sum().reset_index()
        bowl_stats = bowl_stats.merge(mw.groupby('bowler')['is_wicket'].max().reset_index().rename(columns={'is_wicket':'bowl_best_figures'}), on='bowler', how='left')
    
    return bat_stats, bowl_stats, league_name

league_bat_features = {}
league_bowl_features = {}
for folder_name, league_df in all_leagues.items():
    bat_f, bowl_f, lname = compute_league_features(league_df)
    if len(bat_f) > 0:
        bat_f['league'] = lname
        league_bat_features[folder_name] = bat_f
    if len(bowl_f) > 0:
        bowl_f['league'] = lname
        league_bowl_features[folder_name] = bowl_f
    print(f"  {folder_name}: {len(bat_f)} batters, {len(bowl_f)} bowlers qualified")

print(f"\nLeagues with batting data: {len(league_bat_features)}")
print(f"Leagues with bowling data: {len(league_bowl_features)}")

  bbl: 341 batters, 215 bowlers qualified
  blast: 668 batters, 447 bowlers qualified
  bpl: 297 batters, 174 bowlers qualified
  cpl: 264 batters, 155 bowlers qualified
  csat20: 220 batters, 148 bowlers qualified
  hundred: 154 batters, 93 bowlers qualified
  ilt20: 124 batters, 90 bowlers qualified
  ireland: 92 batters, 51 bowlers qualified
  lanka: 118 batters, 72 bowlers qualified
  mlc: 95 batters, 54 bowlers qualified
  npl: 102 batters, 47 bowlers qualified
  nzt20: 171 batters, 104 bowlers qualified
  psl: 215 batters, 130 bowlers qualified
  sa20: 119 batters, 75 bowlers qualified
  sl_domestic: 91 batters, 31 bowlers qualified
  smat: 646 batters, 385 bowlers qualified
  t20i: 2359 batters, 1387 bowlers qualified

Leagues with batting data: 17
Leagues with bowling data: 17


## Cell 4 — League Difficulty Index (batting + bowling calibration)

Calibrates using shared players. Fixed realistic defaults for leagues with few shared players.

In [23]:
ipl_dna = pd.read_csv('../data/player_dna.csv')
ipl_dna['date'] = pd.to_datetime(ipl_dna['date'])
ipl_latest = ipl_dna.sort_values('date').groupby('batsman').last().reset_index()
ipl_players = set(ipl_latest['batsman'].tolist())

# Fixed defaults for leagues with too few shared players
MANUAL_DEFAULTS = {
    'ireland': 0.55,
    'sl_domestic': 0.55,
    'npl': 0.50,
    'lanka': 0.60,
    'nzt20': 0.65,
}

difficulty_index = {}
for folder_name, features in league_bat_features.items():
    # Check for manual override
    if folder_name in MANUAL_DEFAULTS:
        difficulty_index[folder_name] = MANUAL_DEFAULTS[folder_name]
        print(f"  {folder_name}: manual default → {MANUAL_DEFAULTS[folder_name]}")
        continue
    
    shared = set(features['batsman'].tolist()) & ipl_players
    if len(shared) < 5:
        default = MANUAL_DEFAULTS.get(folder_name, 0.75)
        difficulty_index[folder_name] = default
        print(f"  {folder_name}: only {len(shared)} shared players → default {default}")
        continue
    
    ipl_avgs = ipl_latest[ipl_latest['batsman'].isin(shared)][['batsman','career_avg']].rename(columns={'career_avg':'ipl_avg'})
    league_avgs = features[features['batsman'].isin(shared)][['batsman','career_avg']].rename(columns={'career_avg':'league_avg'})
    merged = ipl_avgs.merge(league_avgs, on='batsman')
    merged = merged[(merged['ipl_avg'] > 5) & (merged['league_avg'] > 5)]
    
    if len(merged) < 3:
        difficulty_index[folder_name] = 0.75
        print(f"  {folder_name}: too few valid → default 0.75")
        continue
    
    ratio = np.clip((merged['ipl_avg'] / merged['league_avg']).median(), 0.4, 1.1)
    difficulty_index[folder_name] = round(ratio, 3)
    print(f"  {folder_name}: {len(merged)} shared players → ratio {ratio:.3f}")

print(f"\nLeague Difficulty Index (1.0 = IPL level):")
for league, ratio in sorted(difficulty_index.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(ratio * 20)
    print(f"  {league:12s}: {ratio:.3f}  {bar}")

  bbl: 117 shared players → ratio 0.756
  blast: 103 shared players → ratio 0.628
  bpl: 65 shared players → ratio 0.669
  cpl: 96 shared players → ratio 0.826
  csat20: 44 shared players → ratio 0.667
  hundred: 63 shared players → ratio 0.928
  ilt20: 49 shared players → ratio 0.673
  ireland: manual default → 0.55
  lanka: manual default → 0.6
  mlc: 45 shared players → ratio 0.741
  npl: manual default → 0.5
  nzt20: manual default → 0.65
  psl: 69 shared players → ratio 0.755
  sa20: 43 shared players → ratio 0.968
  sl_domestic: manual default → 0.55
  smat: 155 shared players → ratio 0.625
  t20i: 227 shared players → ratio 0.801

League Difficulty Index (1.0 = IPL level):
  sa20        : 0.968  ███████████████████
  hundred     : 0.928  ██████████████████
  cpl         : 0.826  ████████████████
  t20i        : 0.801  ████████████████
  bbl         : 0.756  ███████████████
  psl         : 0.755  ███████████████
  mlc         : 0.741  ██████████████
  ilt20       : 0.673  ███████

## Cell 5 — Adjust cross-league stats and merge into scouting profiles

In [24]:
ipl_profiles = pd.read_csv('../data/scouting_profiles.csv')
ipl_player_set = set(ipl_profiles['player'].tolist())
print(f"Existing profiles: {len(ipl_profiles)}")

new_players = []
for folder_name, features in league_bat_features.items():
    ratio = difficulty_index.get(folder_name, 0.75)
    for _, row in features.iterrows():
        player = row['batsman']
        if player in ipl_player_set: continue
        adjusted = {
            'player': player,
            'career_avg': round(row['career_avg'] * ratio, 2),
            'career_sr': round(row['career_sr'] * min(ratio*1.05, 1.0), 2),
            'boundary_rate': round(row.get('boundary_rate',0) * min(ratio*1.02, 1.0), 4),
            'peak_score': int(row.get('peak_score',0) * ratio),
            'consistency': row.get('consistency', 15),
            'avg_runs_Powerplay': round(row.get('avg_runs_Powerplay',0) * ratio, 2),
            'avg_runs_Middle': round(row.get('avg_runs_Middle',0) * ratio, 2),
            'avg_runs_Death': round(row.get('avg_runs_Death',0) * ratio, 2),
            'avg_sr_Powerplay': round(row.get('avg_sr_Powerplay',0) * min(ratio*1.05,1.0), 2),
            'avg_sr_Middle': round(row.get('avg_sr_Middle',0) * min(ratio*1.05,1.0), 2),
            'avg_sr_Death': round(row.get('avg_sr_Death',0) * min(ratio*1.05,1.0), 2),
            'matches_played': int(row['innings']),
            'source_league': row.get('league', folder_name),
            'status': 'International', 'current_franchise': 'International',
            'nationality': 'Pakistan' if folder_name == 'psl' else 'Overseas',
        }
        new_players.append(adjusted)

# Add bowlers from other leagues
for folder_name, features in league_bowl_features.items():
    ratio = difficulty_index.get(folder_name, 0.75)
    for _, row in features.iterrows():
        player = row['bowler']
        if player in ipl_player_set: continue
        # Check if already added as batter
        existing = [p for p in new_players if p['player'] == player]
        if existing:
            # Add bowling stats to existing entry
            existing[0]['bowl_wickets'] = int(row.get('bowl_wickets', 0))
            existing[0]['bowl_economy'] = round(row.get('bowl_economy', 8) * (1/min(ratio*1.05,1.0)), 2)  # invert: harder league = better eco
            existing[0]['bowl_avg'] = round(row.get('bowl_avg', 25) * (1/ratio), 2)
            existing[0]['bowl_dot_pct'] = round(row.get('bowl_dot_pct', 35), 1)
            existing[0]['bowl_economy_de'] = round(row.get('bowl_economy_de', 9) * (1/min(ratio*1.05,1.0)), 2)
            existing[0]['bowl_best_figures'] = int(row.get('bowl_best_figures', 0))
        else:
            # Pure bowler not yet in the list
            adjusted = {
                'player': player,
                'career_avg': 0, 'career_sr': 0,  # no batting data
                'bowl_wickets': int(row.get('bowl_wickets', 0)),
                'bowl_economy': round(row.get('bowl_economy', 8) * (1/min(ratio*1.05,1.0)), 2),
                'bowl_avg': round(row.get('bowl_avg', 25) * (1/ratio), 2),
                'bowl_dot_pct': round(row.get('bowl_dot_pct', 35), 1),
                'bowl_economy_po': round(row.get('bowl_economy_po', 8) * (1/min(ratio*1.05,1.0)), 2) if 'bowl_economy_po' in row else None,
                'bowl_economy_mi': round(row.get('bowl_economy_mi', 8) * (1/min(ratio*1.05,1.0)), 2) if 'bowl_economy_mi' in row else None,
                'bowl_economy_de': round(row.get('bowl_economy_de', 9) * (1/min(ratio*1.05,1.0)), 2) if 'bowl_economy_de' in row else None,
                'bowl_best_figures': int(row.get('bowl_best_figures', 0)),
                'matches_played': int(row.get('bowl_matches', 0)),
                'source_league': row.get('league', folder_name),
                'status': 'International', 'current_franchise': 'International',
                'is_bowler': True,
            }
            new_players.append(adjusted)

new_df = pd.DataFrame(new_players)
if len(new_df) > 0:
    new_df = new_df.sort_values('matches_played', ascending=False).drop_duplicates('player', keep='first')
    print(f"New cross-league players: {len(new_df)}")
    bowlers_added = new_df[new_df.get('bowl_wickets', pd.Series(dtype=float)).notna() & (new_df.get('bowl_wickets', pd.Series(dtype=float)) > 0)].shape[0] if 'bowl_wickets' in new_df.columns else 0
    print(f"  With bowling data: {bowlers_added}")
    print(f"\nTop cross-league batters:")
    print(new_df.nlargest(8, 'career_avg')[['player','career_avg','career_sr','source_league','matches_played']].to_string(index=False))

# Count how many leagues each new player appeared in
player_leagues = {}
for folder_name, features in league_bat_features.items():
    for name in features['batsman'].unique():
        if name not in player_leagues:
            player_leagues[name] = set()
        player_leagues[name].add(folder_name)

# Flag PSL-only players (likely Pakistani)
new_df['is_psl_only'] = new_df['player'].apply(
    lambda x: player_leagues.get(x, set()) == {'psl'}
)
print(f"PSL-only players (likely Pakistani): {new_df['is_psl_only'].sum()}")

# Remove them
new_df = new_df[~new_df['is_psl_only']].drop(columns=['is_psl_only'])
print(f"Remaining after removing PSL-only: {len(new_df)}")

# Load BCCI names for Indian detection
people = pd.read_csv('../data/people.csv')
bcci_names_set = set(people[people['key_bcci'].notna()]['name'].tolist())

# Tag nationality: check BCCI key first, then fuzzy match
from rapidfuzz import process, fuzz
bcci_list = list(bcci_names_set)

def tag_nationality(player_name):
    if player_name in bcci_names_set:
        return 'Indian'
    result = process.extractOne(player_name, bcci_list, scorer=fuzz.WRatio, score_cutoff=82)
    if result:
        return 'Indian'
    return 'Overseas'

new_df['nationality'] = new_df['player'].apply(tag_nationality)

# Remove PSL-only players (likely Pakistani)
player_leagues = {}
for folder_name, features in league_bat_features.items():
    for name in features['batsman'].unique():
        if name not in player_leagues:
            player_leagues[name] = set()
        player_leagues[name].add(folder_name)

if len(new_df) > 0:
    new_df['is_psl_only'] = new_df['player'].apply(
        lambda x: player_leagues.get(x, set()) == {'psl'}
    )
    print(f"PSL-only players removed: {new_df['is_psl_only'].sum()}")
    new_df = new_df[~new_df['is_psl_only']]
    if 'is_psl_only' in new_df.columns:
        new_df = new_df.drop(columns=['is_psl_only'])
    print(f"Remaining: {len(new_df)}")
else:
    print("No new players to filter")
print(new_df['nationality'].value_counts())

Existing profiles: 4749
New cross-league players: 47
  With bowling data: 21

Top cross-league batters:
           player  career_avg  career_sr         source_league  matches_played
    Gulraiz Sadaf       27.69      72.07 Pakistan Super League               5
     Saud Shakeel       23.49     103.24 Pakistan Super League              21
     Tayyab Tahir       23.49     115.01 Pakistan Super League              11
    Muhammad Musa       22.27      91.71 Pakistan Super League               7
Abdullah Shafique       21.79     111.11 Pakistan Super League              39
      Umar Siddiq       21.27      89.32 Pakistan Super League               7
        Azhar Ali       19.74      84.35 Pakistan Super League               7
    Sohail Akhtar       19.06      97.64 Pakistan Super League              40
PSL-only players (likely Pakistani): 47
Remaining after removing PSL-only: 0
No new players to filter
Series([], Name: count, dtype: int64)


## Cell 6 — Rebuild scouting profiles with all players

In [ ]:
BATTING_DNA = [
    'career_avg','career_sr','avg_last5','sr_last5','avg_last10','sr_last10',
    'form_score','form_sr','consistency','peak_score','boundary_rate',
    'avg_runs_Death','avg_runs_Middle','avg_runs_Powerplay',
    'avg_sr_Death','avg_sr_Middle','avg_sr_Powerplay',
    'pace_sr','spin_sr','spin_weakness','pace_weakness','high_pressure_avg'
]
BOWLING_DNA = [
    'bowl_wickets','bowl_economy','bowl_avg','bowl_sr','bowl_dot_pct',
    'bowl_economy_po','bowl_economy_mi','bowl_economy_de',
    'bowl_wickets_po','bowl_wickets_mi','bowl_wickets_de',
    'bowl_best_figures','bowl_consistency','bowl_form_economy','bowl_form_wickets',
]

# Fill proxy values for missing DNA features
for col in BATTING_DNA + BOWLING_DNA:
    if col not in new_df.columns:
        if col in ['avg_last5','avg_last10']: new_df[col] = new_df.get('career_avg', 0)
        elif col in ['sr_last5','sr_last10','form_sr','pace_sr','spin_sr']: new_df[col] = new_df.get('career_sr', 0)
        elif col == 'form_score': new_df[col] = new_df.get('career_avg', 0) * 0.8
        elif col == 'high_pressure_avg': new_df[col] = new_df.get('career_avg', 0) * 0.7
        else: new_df[col] = 0

combined = pd.concat([ipl_profiles, new_df], ignore_index=True, sort=False)
combined = combined.drop_duplicates('player', keep='first')
# Fill player_type for cross-league players who don't have it
missing_type = combined['player_type'].isna()

if missing_type.any():
    def assign_player_type(row):
        has_bowling = pd.notna(row.get('bowl_wickets')) and row.get('bowl_wickets', 0) >= 20
        has_batting = pd.notna(row.get('career_avg')) and row.get('career_avg', 0) > 10
        if has_batting and has_bowling:
            return 'All-Rounder'
        elif has_bowling:
            return 'Bowler'
        else:
            return 'Batsman'
    
    combined.loc[missing_type, 'player_type'] = combined[missing_type].apply(assign_player_type, axis=1)

# Fill batting_role for non-bowlers who don't have it
missing_bat_role = combined['batting_role'].isna() & (combined['player_type'] != 'Bowler')

if missing_bat_role.any():
    def assign_batting_role(row):
        sr = row.get('career_sr', 0) or 0
        avg = row.get('career_avg', 0) or 0
        death_sr = row.get('avg_sr_Death', 0) or 0
        if sr > 140 and death_sr > 145:
            return 'Finisher'
        elif avg > 30 and sr < 125:
            return 'Opener'
        elif avg > 25:
            return 'Top Order'
        else:
            return 'Middle Order'
    
    combined.loc[missing_bat_role, 'batting_role'] = combined[missing_bat_role].apply(assign_batting_role, axis=1)

# Fill bowling_category for bowlers/ARs who don't have it
missing_bowl = combined['bowling_category'].isna() & (combined['player_type'].isin(['Bowler', 'All-Rounder']))

if missing_bowl.any():
    def assign_bowling_cat(row):
        style = row.get('bowling_style')
        if pd.notna(style):
            style = str(style)
            if style in ['RFM', 'RF', 'LFM', 'LF']:
                return 'Fast'
            elif style == 'OB':
                return 'Off Spin'
            elif style in ['LB', 'SLA', 'SLAC']:
                return 'Leg Spin'
        return None  # unknown bowling style
    
    combined.loc[missing_bowl, 'bowling_category'] = combined[missing_bowl].apply(assign_bowling_cat, axis=1)

# Fill bowling_specialty for bowlers/ARs who don't have it
missing_spec = combined['bowling_specialty'].isna() & (combined['player_type'].isin(['Bowler', 'All-Rounder']))

if missing_spec.any():
    def assign_bowling_spec(row):
        eco = row.get('bowl_economy', 99) or 99
        eco_pp = row.get('bowl_economy_po', 99) or 99
        eco_mi = row.get('bowl_economy_mi', 99) or 99
        eco_de = row.get('bowl_economy_de', 99) or 99
        dot = row.get('bowl_dot_pct', 0) or 0
        best = min(eco_pp, eco_mi, eco_de)
        if eco < 7.0 and dot > 40:
            return 'Defensive (Run Stopper)'
        if best == eco_de and eco_de < 8.5:
            return 'Death Specialist'
        if best == eco_pp and eco_pp < 7.5:
            return 'Powerplay Specialist'
        if best == eco_mi and eco_mi < 7.5:
            return 'Middle Over Specialist'
        return 'Wicket-taker (Strike Bowler)'
    
    combined.loc[missing_spec, 'bowling_specialty'] = combined[missing_spec].apply(assign_bowling_spec, axis=1)

print("After filling cross-league metadata:")
print(f"  player_type NaN: {combined['player_type'].isna().sum()}")
print(f"  batting_role NaN: {combined['batting_role'].isna().sum()}")
print(f"  bowling_category NaN: {combined['bowling_category'].isna().sum()}")
print(combined['player_type'].value_counts())
# Flags
combined['is_bowler'] = combined['is_bowler'].fillna(False) | ((combined['bowl_wickets'].fillna(0) >= 20))
combined['is_allrounder'] = (combined['career_avg'].fillna(0) > 12) & combined['is_bowler']

# Recompute all scores
def normalize_col(s):
    mn, mx = s.quantile(0.05), s.quantile(0.95)
    if mx == mn: return pd.Series(50.0, index=s.index)
    return ((s - mn) / (mx - mn) * 100).clip(0, 100)

combined['bat_performance'] = (
    combined['career_avg'].fillna(0)*0.25 + combined['career_sr'].fillna(0)*0.15 +
    combined['form_score'].fillna(0)*0.20 + combined['boundary_rate'].fillna(0)*100*0.10 +
    combined['avg_sr_Death'].fillna(0)*0.10 + combined['high_pressure_avg'].fillna(0)*0.10 +
    combined['peak_score'].fillna(0)*0.10)
combined['bowl_performance'] = 0.0
bm = combined['bowl_wickets'].fillna(0) > 0
combined.loc[bm,'bowl_performance'] = (
    combined.loc[bm,'bowl_wickets'].fillna(0)*0.20 +
    (12-combined.loc[bm,'bowl_economy'].clip(upper=12).fillna(8))*10*0.20 +
    combined.loc[bm,'bowl_dot_pct'].fillna(35)*0.15 +
    (12-combined.loc[bm,'bowl_economy_de'].clip(upper=12).fillna(8))*10*0.15 +
    combined.loc[bm,'bowl_best_figures'].fillna(0)*8*0.10 +
    combined.loc[bm,'bowl_wickets_de'].fillna(0)*100*0.10 +
    (30-combined.loc[bm,'bowl_sr'].clip(upper=30).fillna(20))*2*0.10)

def combined_perf(row):
    bat, bowl = row['bat_performance'], row['bowl_performance']
    if row.get('is_allrounder',False): return max(bat,bowl)*0.6 + min(bat,bowl)*0.4
    elif row.get('is_bowler',False): return bowl
    return bat
combined['performance_score'] = combined.apply(combined_perf, axis=1)
pmin,pmax = combined['performance_score'].quantile(0.05), combined['performance_score'].quantile(0.95)
combined['performance_rating'] = ((combined['performance_score']-pmin)/(pmax-pmin)*100).clip(0,100).round(1)
combined['form_rating'] = normalize_col(combined['form_score'].fillna(0)).round(1)
combined['consistency_rating'] = (100-normalize_col(combined['consistency'].fillna(15))).round(1)
combined['impact_rating'] = normalize_col(combined['peak_score'].fillna(0)*0.6+combined['boundary_rate'].fillna(0)*100*0.4).round(1)
combined['pressure_rating'] = normalize_col(combined['high_pressure_avg'].fillna(0)).round(1)
combined['scouting_score'] = (combined['performance_rating']*0.30 + combined['form_rating']*0.25 +
    combined['consistency_rating']*0.15 + combined['impact_rating']*0.15 + combined['pressure_rating']*0.15).round(1)

def classify_playstyle(row):
    sr=row.get('career_sr',0) or 0; avg=row.get('career_avg',0) or 0
    br=row.get('boundary_rate',0) or 0; dsr=row.get('avg_sr_Death',0) or 0
    is_b=row.get('is_bowler',False); is_ar=row.get('is_allrounder',False)
    beco=row.get('bowl_economy',99) or 99; bdot=row.get('bowl_dot_pct',0) or 0
    bde=row.get('bowl_economy_de',99) or 99
    if is_ar: return 'Batting Allrounder' if avg>20 or sr>135 else 'Bowling Allrounder'
    if is_b:
        if bde<8.0 and beco<7.5: return 'Death Specialist'
        elif beco<7.0 and bdot>45: return 'Restrictive Bowler'
        elif beco<8.5: return 'Wicket-taker'
        return 'Support Bowler'
    if sr>145 and br>0.17: return 'Aggressive'
    elif avg>25 and sr<125: return 'Anchor'
    elif dsr>145 and sr>130: return 'Finisher'
    elif avg>20 and sr>125: return 'Impact'
    return 'Utility'
combined['playstyle'] = combined.apply(classify_playstyle, axis=1)

intl = combined[combined['status']=='International']
print(f"Combined: {len(combined)} total ({len(intl)} from other leagues)")
if len(intl)>0:
    print(f"\nTop 10 international scouting picks:")
    print(intl.nlargest(10,'scouting_score')[['player','scouting_score','career_avg','career_sr','bowl_wickets','bowl_economy','playstyle','source_league']].to_string(index=False))

After filling cross-league metadata:
  player_type NaN: 0
  batting_role NaN: 728
  bowling_category NaN: 3643
player_type
Batsman          3617
Bowler            728
All-Rounder       378
Wicket Keeper      26
Name: count, dtype: int64
Combined: 4749 total (4045 from other leagues)

Top 10 international scouting picks:
       player  scouting_score  career_avg  career_sr  bowl_wickets  bowl_economy  playstyle                      source_league
      JJ Smit            87.9       27.52     123.12           NaN           NaN     Anchor Sri Lanka in Australia T20I Series
  D Potgieter            86.5       27.10     140.00           NaN           NaN   Finisher                               SA20
     DS Airee            86.4       26.13     118.50           NaN           NaN     Anchor Sri Lanka in Australia T20I Series
    RA Herman            86.1       31.83     119.70           NaN           NaN     Anchor                               SA20
 Manan Bashir            85.9       28.20  

## Cell 7 — Save everything

In [26]:
ALL_DNA = BATTING_DNA + BOWLING_DNA
dna_matrix = combined[ALL_DNA].fillna(0).values
scaler = MinMaxScaler()
dna_scaled = scaler.fit_transform(dna_matrix)

combined.to_csv('../data/scouting_profiles.csv', index=False)
np.save('../data/dna_matrix_scaled.npy', dna_scaled)
np.save('../data/player_names_index.npy', combined['player'].values)

diff_df = pd.DataFrame([{'league':k,'difficulty_ratio':v} for k,v in difficulty_index.items()]).sort_values('difficulty_ratio', ascending=False)
diff_df.to_csv('../data/league_difficulty_index.csv', index=False)

print(f"✅ Saved scouting_profiles.csv: {len(combined)} players")
print(f"  IPL: {len(combined[combined['status']!='International'])}")
print(f"  International: {len(combined[combined['status']=='International'])}")
print(f"  DNA matrix: {dna_scaled.shape} (37 dimensions)")
print(f"  Difficulty index: {len(diff_df)} leagues saved")

✅ Saved scouting_profiles.csv: 4749 players
  IPL: 704
  International: 4045
  DNA matrix: (4749, 37) (37 dimensions)
  Difficulty index: 17 leagues saved


In [27]:
df = pd.read_csv('../data/scouting_profiles.csv')
print(f"Total rows: {len(df)}")
print(f"\nStatus distribution:")
print(df['status'].value_counts(dropna=False))
print(f"\nColumns present: {'player_type' in df.columns}, {'batting_role' in df.columns}, {'bowling_category' in df.columns}")

Total rows: 4749

Status distribution:
status
International    4045
Retired           466
Active            183
Unsold             55
Name: count, dtype: int64

Columns present: True, True, True


In [28]:
df = pd.read_csv('../data/scouting_profiles.csv')
print(f"Rows: {len(df)}")
print(df['status'].value_counts())

Rows: 4749
status
International    4045
Retired           466
Active            183
Unsold             55
Name: count, dtype: int64


In [29]:
intl = combined[combined['status'] == 'International']
print(f"Total international: {len(intl)}")
if 'source_league' in intl.columns:
    print(intl['source_league'].value_counts())

Total international: 4045
source_league
Sri Lanka in Australia T20I Series                  1972
Syed Mushtaq Ali Trophy                              529
NatWest T20 Blast                                    485
Big Bash League                                      168
CSA T20 Challenge                                    161
Super Smash                                          144
Bangladesh Premier League                            138
Caribbean Premier League                             106
Cricket Ireland Inter-Provincial Twenty20 Trophy      86
Major Clubs T20 Tournament                            75
Pakistan Super League                                 54
Nepal Premier League                                  43
Lanka Premier League                                  36
SA20                                                  20
Major League Cricket                                  18
International League T20                               9
The Hundred Men's Competition                   

In [30]:
df = pd.read_csv('../data/scouting_profiles.csv')
print(f"Total: {len(df)}")
print(df['nationality'].value_counts())

Total: 4749
nationality
Overseas    2520
Indian      2229
Name: count, dtype: int64


In [31]:
print(f"New players tagged Indian: {(new_df['nationality'] == 'Indian').sum()}")
print(f"New players tagged Overseas: {(new_df['nationality'] == 'Overseas').sum()}")

# Check a known SMAT Indian player
smat = new_df[new_df.get('source_league', '').str.contains('Mushtaq', case=False, na=False)]
print(f"\nSMAT players: {len(smat)}")
print(f"SMAT tagged Indian: {(smat['nationality'] == 'Indian').sum()}")
print(f"\nSample SMAT names:")
print(smat['player'].head(10).tolist())
print(f"\nSample BCCI names:")
print(list(bcci_names_set)[:10])

New players tagged Indian: 0
New players tagged Overseas: 0

SMAT players: 0
SMAT tagged Indian: 0

Sample SMAT names:
[]

Sample BCCI names:
['Deepak Gohain', 'Akash Sudan', 'Lee Yong Lepcha', 'Vivekanand Tiwari', 'CK Paul', 'Aamir Gani', 'MB Darshan', 'Rahul Khajan Singh', 'Manish Sharma', 'Zafar Ali']


In [32]:
print(f"new_df rows: {len(new_df)}")
print(f"new_df columns: {list(new_df.columns)[:10]}")

new_df rows: 0
new_df columns: ['player', 'career_avg', 'career_sr', 'boundary_rate', 'peak_score', 'consistency', 'avg_runs_Powerplay', 'avg_runs_Middle', 'avg_runs_Death', 'avg_sr_Powerplay']


In [33]:
df = pd.read_csv('../data/scouting_profiles.csv')
print(f"Total: {len(df)}")
print(df['nationality'].value_counts())

Total: 4749
nationality
Overseas    2520
Indian      2229
Name: count, dtype: int64
